Scenario: Corporate Product Launch Broadcast

Imagine a company preparing to launch Product X in Q3. The Coordinator Agent (like a corporate program manager) sends out a broadcast announcement to all departments at once:
“Product X launch in Q3, target market North America, budget $5M.”


📢 Coordinator Agent (Broadcaster)
- Sends the launch announcement to all departments simultaneously.
- This is implemented in the code by the broadcast() function, which uses asyncio.gather() to run all agents in parallel.

📈 Marketing Agent
- Receives the broadcast.
- Responds with a marketing strategy: campaigns, channels, and positioning.
💰 Finance Agent
- Receives the broadcast.
- Responds with budget allocation and ROI forecasts.
🏭 Operations Agent
- Receives the broadcast.
- Responds with production and supply chain actions.
👩‍⚖️ Legal Agent
- Receives the broadcast.
- Responds with compliance checks and contract actions.
👩‍💼 HR Agent
- Receives the broadcast.
- Responds with staffing and training plans.

🧑‍⚖️ Decision Agent
- Collects all responses.
- Integrates them into a Final Corporate Launch Plan.
- In the code, this is the decision_agent() function that merges all outputs into one consolidated plan.

In [ ]:
!pip install groq

In [4]:
from groq import Groq
import asyncio
from google.colab import userdata
import json

api_key = userdata.get("newapi")

if not api_key:
    raise ValueError("API key not found in Colab Secrets")

client = Groq(api_key=api_key)
MODEL_NAME = "llama-3.3-70b-versatile"

In [5]:
# HELPER FUNCTIONS
def clean_response(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()
    return text


def safe_parse_json(text):
    try:
        return json.loads(text)
    except:
        return {"error": "Invalid JSON", "raw_output": text}


async def call_llm(prompt):
    loop = asyncio.get_event_loop()

    def sync_call():
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Return ONLY valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        content = response.choices[0].message.content

        if not content:
            return {"error": "Empty response"}

        content = clean_response(content)
        return safe_parse_json(content)

    return await loop.run_in_executor(None, sync_call)


In [6]:
async def marketing_agent(message):
    print("\n[Marketing Agent]")

    prompt = f"""
    Launch announcement: {message}

    Provide marketing strategy.

    JSON:
    {{
      "campaigns": [],
      "channels": [],
      "positioning": ""
    }}
    """
    return await call_llm(prompt)


async def finance_agent(message):
    print("\n[Finance Agent]")

    prompt = f"""
    Launch announcement: {message}

    Provide financial plan.

    JSON:
    {{
      "budget_allocation": "",
      "roi_forecast": ""
    }}
    """
    return await call_llm(prompt)


async def operations_agent(message):
    print("\n[Operations Agent]")

    prompt = f"""
    Launch announcement: {message}

    Provide operations plan.

    JSON:
    {{
      "production": "",
      "supply_chain": "",
      "logistics": ""
    }}
    """
    return await call_llm(prompt)


async def legal_agent(message):
    print("\n[Legal Agent]")

    prompt = f"""
    Launch announcement: {message}

    Provide legal compliance plan.

    JSON:
    {{
      "compliance": "",
      "contracts": ""
    }}
    """
    return await call_llm(prompt)


async def hr_agent(message):
    print("\n[HR Agent]")

    prompt = f"""
    Launch announcement: {message}

    Provide HR plan.

    JSON:
    {{
      "hiring": "",
      "training": ""
    }}
    """
    return await call_llm(prompt)

In [7]:
# COORDINATOR Agent (BROADCASTER)
async def broadcast(message):
    print("\n[Coordinator] Broadcasting to all agents...")

    results = await asyncio.gather(
        marketing_agent(message),
        finance_agent(message),
        operations_agent(message),
        legal_agent(message),
        hr_agent(message)
    )

    return {
        "marketing": results[0],
        "finance": results[1],
        "operations": results[2],
        "legal": results[3],
        "hr": results[4],
    }


In [8]:
# DECISION AGENT
async def decision_agent(message, reports):
    print("\n[Decision Agent] Creating final launch plan...")

    prompt = f"""
    Announcement:
    {message}

    Department Reports:
    {json.dumps(reports, indent=2)}

    Create final corporate launch plan.

    JSON:
    {{
      "final_plan": "",
      "key_actions": []
    }}
    """
    return await call_llm(prompt)

In [10]:
# MAIN SYSTEM
async def corporate_launch_system():
    message = "Product X launch in Q3, target market North America, budget $5M"
    print("Announcement:", message)

    # Step 1: Broadcast (parallel execution)
    reports = await broadcast(message)

    # Step 2: Final decision
    final_plan = await decision_agent(message, reports)

    return {
        "announcement": message,
        "reports": reports,
        "final_plan": final_plan
    }


# RUN
result = await corporate_launch_system()

print("\n===== FINAL OUTPUT =====")
print(json.dumps(result, indent=2))

Announcement: Product X launch in Q3, target market North America, budget $5M

[Coordinator] Broadcasting to all agents...

[Marketing Agent]

[Finance Agent]

[Operations Agent]

[Legal Agent]

[HR Agent]

[Decision Agent] Creating final launch plan...

===== FINAL OUTPUT =====
{
  "announcement": "Product X launch in Q3, target market North America, budget $5M",
  "reports": {
    "marketing": {
      "campaigns": [
        {
          "name": "Product X Introduction",
          "objective": "Create awareness and buzz around Product X",
          "target_audience": "Tech-savvy individuals and businesses in North America",
          "budget_allocation": "$1.5M",
          "timeline": "Q2-Q3"
        },
        {
          "name": "Influencer Partnerships",
          "objective": "Partner with influencers to showcase Product X's features and benefits",
          "target_audience": "Influencers in the tech industry with a focus on North America",
          "budget_allocation": "$1M",
  